In [ ]:
df = pd.read_csv("../../Data/Final/TT Split/changi_train_long.csv")
print(df.head())


            x         y          Date      Value
0  103.964566  1.350459  Mar-Apr 2000  23.775064
1  103.964566  1.351269  Mar-Apr 2000  23.403744
2  103.964566  1.352079  Mar-Apr 2000  22.966872
3  103.964566  1.352889  Mar-Apr 2000  22.417757
4  103.965377  1.348838  Mar-Apr 2000  23.699749


In [ ]:
## THIS CODE IS TO REFORMAT THE TIME INDEX IN THE DATASET ##
import pandas as pd

# Function to create a time index
def create_time_index(df):
    bimonthly_map = {"Jan-Feb": 0, "Mar-Apr": 1, "May-Jun": 2, "Jul-Aug": 3, "Sep-Oct": 4, "Nov-Dec": 5}
    
    df["Year"] = df["Date"].str[-4:].astype(int)  # Extract year
    df["Period"] = df["Date"].str[:-5].map(bimonthly_map)  # Extract period and map it

    df["time_index"] = (df["Year"] - 2000) * 6 + df["Period"]  # Compute time index
    df = df.drop(columns=["Year", "Period"])  # Drop extra columns
    
    return df

# Load dataset
changi_test_long = pd.read_csv("../../Data/Final/TT Split/changi_train_long.csv")

# Apply time index conversion
changi_test_long = create_time_index(changi_test_long)

# Save the updated dataset if needed
changi_test_long.to_csv("changi_train_long_time_index.csv", index=False)

# Check first few rows
print(changi_test_long.head())


            x         y          Date      Value  time_index
0  103.964566  1.350459  Mar-Apr 2000  23.775064           1
1  103.964566  1.351269  Mar-Apr 2000  23.403744           1
2  103.964566  1.352079  Mar-Apr 2000  22.966872           1
3  103.964566  1.352889  Mar-Apr 2000  22.417757           1
4  103.965377  1.348838  Mar-Apr 2000  23.699749           1


In [ ]:
import pandas as pd
import numpy as np

# Load the dataset
df = pd.read_csv("changi_train_long_time_index.csv")

# Define sequence length
sequence_length = 24

# Group by (x, y) and process each time series separately
grouped = df.groupby(["x", "y"])

# List to store sequences
all_sequences = []

# Iterate over each coordinate group
for (x, y), group in grouped:
    # Sort by time index
    group = group.sort_values(by="time_index")

    # Extract LST values
    values = group["Value"].values

    # Generate sequences
    for i in range(len(values) - sequence_length):
        input_seq = values[i : i + sequence_length]  # 24 time steps as input
        target_value = values[i + sequence_length]   # Next value as target
        all_sequences.append([x, y] + list(input_seq) + [target_value])

# Convert to DataFrame
columns = ["x", "y"] + [f"LST_t-{i}" for i in range(sequence_length, 0, -1)] + ["Target"]
sequences_df = pd.DataFrame(all_sequences, columns=columns)

# Save as CSV
sequences_df.to_csv("changi_train_sequences.csv", index=False)

# Print sample sequences
print(sequences_df.head())


            x         y   LST_t-24   LST_t-23   LST_t-22   LST_t-21  \
0  103.964566  1.350459  23.775064  27.537897  27.519318  32.314507   
1  103.964566  1.350459  27.537897  27.519318  32.314507  23.391225   
2  103.964566  1.350459  27.519318  32.314507  23.391225  29.400195   
3  103.964566  1.350459  32.314507  23.391225  29.400195  29.712163   
4  103.964566  1.350459  23.391225  29.400195  29.712163  27.845558   

    LST_t-20   LST_t-19   LST_t-18   LST_t-17  ...    LST_t-9    LST_t-8  \
0  23.391225  29.400195  29.712163  27.845558  ...  20.002272  29.542257   
1  29.400195  29.712163  27.845558  24.015326  ...  29.542257  26.961560   
2  29.712163  27.845558  24.015326  25.797058  ...  26.961560  30.824972   
3  27.845558  24.015326  25.797058  29.891817  ...  30.824972  22.669133   
4  24.015326  25.797058  29.891817  31.118223  ...  22.669133  27.795971   

     LST_t-7    LST_t-6    LST_t-5    LST_t-4    LST_t-3    LST_t-2  \
0  26.961560  30.824972  22.669133  27.795971